In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay)

In [ ]:
# 1. Carga y Preparación del Dataset Transversal (oasis_cross-sectional.csv)
df = pd.read_csv('oasis_cross-sectional.csv')

# Selección de atributos según el informe_2
features = ['Sex', 'Age', 'Educ', 'SES', 'MMSE', 'eTIV', 'nWBV', 'ASF']
X = df[features].copy()

# Codificación de Sexo (M=0, F=1) para scikit-learn
X['Sex'] = X['Sex'].map({'M': 0, 'F': 1})

# --- LINEAMIENTO: Binarización de CDR (2 Clases) ---
# CDR 0 -> 0 (Non demented), CDR >= 0.5 -> 1 (Demented)
y = df['CDR'].apply(lambda x: 1 if x >= 0.5 else 0)

In [ ]:
# 2. Tratamiento de Datos Faltantes (SES y Educ tienen nulos en este set)
# Usamos imputación por mediana para no perder filas valiosas [1]
imputer = SimpleImputer(strategy='median')
X_imputed = imputer.fit_transform(X)

# 3. Configuración Experimental (10 Validaciones 80/20)
results = {'acc': [], 'prec': [], 'rec': [], 'f1': [], 'auc': []}
fig, axes = plt.subplots(2, 5, figsize=(22, 10))
axes = axes.flatten()

print("Iniciando 10 validaciones independientes para Árbol de Decisión...")

for i in range(10):
    # Partición aleatoria única por iteración (Sección 12 de la guía) [1]
    X_train, X_test, y_train, y_test = train_test_split(
        X_imputed, y, test_size=0.2, random_state=i
    )
    
    # 4. Entrenamiento del Árbol de Decisión con Control de Complejidad
    # Usamos max_depth=4 para evitar que el árbol "memorice" el ruido (sobreajuste) [1]
    # class_weight='balanced' para compensar el desbalance natural entre clases
    dt = DecisionTreeClassifier(max_depth=4, 
                                min_samples_leaf=10, 
                                class_weight='balanced', 
                                random_state=42)
    dt.fit(X_train, y_train)
    
    # Predicciones
    y_pred = dt.predict(X_test)
    y_proba = dt.predict_proba(X_test)[:, 1]
    
    # 5. Registro de Métricas (Sección 13 de la guía) [1]
    results['acc'].append(accuracy_score(y_test, y_pred))
    results['prec'].append(precision_score(y_test, y_pred))
    results['rec'].append(recall_score(y_test, y_pred))
    results['f1'].append(f1_score(y_test, y_pred))
    results['auc'].append(roc_auc_score(y_test, y_proba))
    
    # 6. Generación de la Matriz de Confusión para cada iteración [Instrucción de estilo]
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Dem.', 'Dem.'])
    disp.plot(ax=axes[i], cmap='Greens', colorbar=False)
    axes[i].set_title(f"Iteración {i+1}\nF1-Score: {results['f1'][-1]:.3f}")

plt.suptitle("Matrices de Confusión (2 Clases) - Árboles de Decisión Transversal", fontsize=18)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

# 7. Reporte Final de Desempeño (Media ± DE)
print("\n" + "="*50)
print(f"--- DESEMPEÑO FINAL ÁRBOL DE DECISIÓN ---")
print(f"{'Métrica':<20} | {'Promedio ± DE (10 iteraciones)':<20}")
print("-" * 50)
for m in results:
    print(f"{m.upper():<20} | {np.mean(results[m]):.4f} ± {np.std(results[m]):.4f}")
print("="*50)